Read data from Bronze Table:

In [0]:
from pyspark.sql.functions import trim, col, when
from pyspark.sql.types import StringType

In [0]:
#df = spark.read.table('workspace.bronze.crm_cust_info')
df = spark.sql("SELECT * FROM workspace.bronze.crm_cust_info")


Do Transformations of Data:

In [0]:
df= df.filter(col('cst_id').isNotNull())



In [0]:


# Trim all string columns
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        #df = df.withColumn('str_' + field.name, trim(col(field.name)))
        df = df.withColumn(field.name, trim(col(field.name)))

#display(df)       

In [0]:
'''
df = (
        df.withColumn('cst_marital_status',
                        when(col('cst_marital_status') == 'M', "MARRIED")
                        .when(col('cst_marital_status') == 'S', "SINGLE")
                        .otherwise('UNKNOWN'),                                            
                    )
            .withColumn('cst_gndr',
                        when(col('cst_gndr') == 'M', "Male")
                        .when(col('cst_gndr') == 'F', "Female")
                        .otherwise('UNKNOWN')      
                        )
)

'''
# More efficient way of compbining multiple transformations:
df = df.withColumns({
        'cst_marital_status': when(col('cst_marital_status') == 'M', "MARRIED")
                             .when(col('cst_marital_status') == 'S',"SINGLE")
                             .otherwise('Unknown'),
        'cst_gndr': when(col('cst_gndr') == 'M', "Male")
                    .when(col('cst_gndr') == 'F',"Female")
                    .otherwise('Unknown')

    })
#display(df)

In [0]:
# Rename Columns:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
display(df)    
#df.write.mode('overwrite').saveAsTable()

In [0]:
# Save transformed data in Delta table:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")

In [0]:
%sql
Select * from workspace.silver.crm_customers